# 04 — Train RT-DETR

Trains RT-DETR-Medium on the BDD100K clear-weather training split.
RT-DETR is a full transformer detector (COCO pretrained, no foundation model).
Checkpoints are saved every 10 epochs to `checkpoints/rtdetr/`.

**Note:** RT-DETR uses the Ultralytics API via `RTDETR()`, not `YOLO()`.

**Prerequisites:** run `01_setup_and_data.ipynb` first.

In [ ]:
from pathlib import Path
import torch
from dotenv import load_dotenv
from ultralytics import RTDETR
from src.train_utils import setup_logging

DRIVE_ROOT  = Path('/content/drive/MyDrive/FON/master_rad/bdd100k_data')
CONFIGS_DIR = Path('/content/data_prepared/configs')
CHECKPOINTS = DRIVE_ROOT / 'checkpoints'

load_dotenv(DRIVE_ROOT / '.env')
logger = setup_logging()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
model = RTDETR('rtdetr-m.pt')

In [ ]:
results = model.train(
    data=str(CONFIGS_DIR / 'dataset.yaml'),
    epochs=100,
    batch=16,
    imgsz=640,
    optimizer='AdamW',
    lr0=0.0001,
    lrf=0.01,
    warmup_epochs=5,
    cos_lr=True,
    device=DEVICE,
    project=str(CHECKPOINTS / 'rtdetr'),
    name='run',
    save_period=10,
    exist_ok=True,
    plots=True,
)
print('Training complete.')
print('Best mAP@50:', results.results_dict.get('metrics/mAP50(B)'))

In [ ]:
import json

summary = {
    'model': 'rtdetr',
    'best_map50': results.results_dict.get('metrics/mAP50(B)'),
    'best_map50_95': results.results_dict.get('metrics/mAP50-95(B)'),
    'epochs': 100,
}
out = CHECKPOINTS / 'rtdetr' / 'training_summary.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(summary, indent=2))
print('Summary saved to', out)